# Concrete Evidence: train the rebar segmentation model (YOLOv8-seg)

**Before running:** Runtime → Change runtime type → **T4 GPU**.

- **Part A, the baseline:** our own labelled cage frames (`bar · tie · hook_90 · hook_135`, from Roboflow), starting from the standard pretrained YOLOv8 weights. These are the numbers for the slides.
- **Part B, optional warm-up:** train on **ConRebSeg** first, then on our frames, with the same split and seed. Keep it only if it beats Part A on the same test frames.

Why our frames are the baseline: no open dataset outlines single bars in a cage before the pour. ConRebSeg (DTU, CC BY 4.0) is real site photos of concrete *repair*, and its labels outline *patches* of exposed rebar, so here they're called `exposed_rebar`, never `bar`. Column Tie Detection and SteelBindings have boxes only.

In [ ]:
# 1. GPU + install (same versions as requirements.txt)
!nvidia-smi -L
!pip install -q ultralytics==8.4.216 roboflow==1.7.3

In [ ]:
# 2. Results go to Google Drive, so best.pt survives the session
from google.colab import drive
drive.mount('/content/drive')
SAVE = '/content/drive/MyDrive/ConcreteEvidence/runs'
import os; os.makedirs(SAVE, exist_ok=True)
SEED, IMGSZ = 0, 640  # the same for every run, so the runs compare

In [ ]:
# 3. The repo code
%cd /content
!rm -rf AAD_Research_Studio_03 && git clone -q https://github.com/iaac-maai/AAD_Research_Studio_03
%cd AAD_Research_Studio_03

## Part A: baseline on our own frames

In Roboflow: **Generate** a version (70/20/10 split, blur + brightness augmentation). Type the API key into the box below; it is never saved in the notebook.

In [ ]:
# 4. Our labelled frames from Roboflow
from getpass import getpass
from roboflow import Roboflow
VERSION = 1
RF_WORKSPACE = 'diegos-workspace-puwbt'              # where the labels live
RF_PROJECT = 'concrete-evidence-column-cages-ft8gc'
rf = Roboflow(api_key=getpass('Roboflow API key: '))
proj = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
proj.version(VERSION).download('yolov8', location='data/ours', overwrite=True)
!cat data/ours/data.yaml

In [ ]:
# 5. Look at one labelled frame before training
import glob, random, cv2, numpy as np, matplotlib.pyplot as plt
def show_labels(pattern):  # labels sit next to images: .../images/... -> .../labels/....txt
    img = random.choice(glob.glob(pattern))
    lbl = img.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    im = cv2.cvtColor(cv2.imread(img), cv2.COLOR_BGR2RGB); h, w = im.shape[:2]
    for line in open(lbl).read().splitlines():
        if not line.strip(): continue
        v = list(map(float, line.split())); pts = (np.array(v[1:]).reshape(-1, 2) * [w, h]).astype(np.int32)
        cv2.polylines(im, [pts], True, (255, 0, 0), 3)
    plt.figure(figsize=(12, 8)); plt.imshow(im); plt.axis('off'); plt.title(os.path.basename(img))
show_labels('data/ours/train/images/*')

In [ ]:
# 6. TRAIN the baseline from the standard pretrained weights
from ultralytics import YOLO
def train(start, data, name, epochs):
    YOLO(start).train(data=data, epochs=epochs, imgsz=IMGSZ, batch=8, patience=20, seed=SEED,
                      deterministic=True, max_det=500, project=SAVE, name=name, exist_ok=True)
    return f'{SAVE}/{name}/weights/best.pt'
baseline = train('yolov8n-seg.pt', 'data/ours/data.yaml', 'ours_baseline', epochs=100)

In [ ]:
# 7. Score on the test frames (never seen in training) -> the board and the slides
def score(weights, data='data/ours/data.yaml'):
    m = YOLO(weights).val(data=data, split='test', imgsz=IMGSZ, max_det=500, plots=False)
    print(f'{weights}\n  mask mAP50 {m.seg.map50:.4f}   precision {m.seg.mp:.4f}   recall {m.seg.mr:.4f}')
    return m
score(baseline)
for r in YOLO(baseline).predict(sorted(glob.glob('data/ours/test/images/*'))[:3], imgsz=IMGSZ, conf=0.25, max_det=500):
    plt.figure(figsize=(12, 8)); plt.imshow(r.plot()[:, :, ::-1]); plt.axis('off')

## Part B (optional): ConRebSeg warm-up

22.6 GB straight to Colab's disk (not your Drive), a few minutes, every file checked against the published checksums. Then about 40-60 min of training on a free T4. Skip this part if time is short.

Credit: Schmidt & Nalpantidis (2025), *Segmentation dataset for reinforced concrete construction*, Automation in Construction 171. Data DOI 10.11583/DTU.26213762.

In [ ]:
# 8. Download ConRebSeg from figshare (DTU Data runs on figshare) and check it
import hashlib, subprocess
D = 'data/conrebseg'; os.makedirs(D, exist_ok=True)
FILES = {47627011: 'metadata.json', 47627014: 'samples.json', 47836036: 'ConRebSeg.zip'}
MD5 = {'metadata.json': '8024f05766f52549abfbf6e869679ce3', 'samples.json': '3a7ca34b89077fd6104a4fa8a5aba0b5',
       'ConRebSeg.zip': 'c1517d9e41fc59cc7be2835b3e2f4642'}
for fid, name in FILES.items():
    subprocess.run(['curl', '-sSL', '--retry', '10', '-C', '-', '-o', f'{D}/{name}',
                    f'https://ndownloader.figshare.com/files/{fid}'], check=True)
    h = hashlib.md5()
    with open(f'{D}/{name}', 'rb') as f:
        for block in iter(lambda: f.read(1 << 24), b''): h.update(block)
    assert h.hexdigest() == MD5[name], f'{name} is damaged: download it again'
    print('ok', name)

In [ ]:
# 9. Unpack the site photos (no YouTube), convert (class exposed_rebar), look at one
!python pipeline.py conrebseg
!rm -f data/conrebseg/ConRebSeg.zip
show_labels('data/conrebseg_yolo/images/train/*')

In [ ]:
# 10. Warm up on ConRebSeg, then train on our frames with the same seed, and compare with Part A
warm = train('yolov8n-seg.pt', 'data/conrebseg_yolo/data.yaml', 'conrebseg_warmup', epochs=20)
warmed = train(warm, 'data/ours/data.yaml', 'ours_after_warmup', epochs=100)
print('Part A (baseline):'); score(baseline)
print('Part B (after warm-up):'); score(warmed)
print('Keep the warm-up only if its mask mAP50 is higher. Log both runs on the team board.')